# Day 4

## Tokenizing with code

In [5]:
pip install tiktoken

   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   ---- ----------------------------------- 92.2/874.7 kB 1.7 MB/s eta 0:00:01
   -------------- ------------------------- 307.2/874.7 kB 3.2 MB/s eta 0:00:01
   ------------------- -------------------- 430.1/874.7 kB 3.4 MB/s eta 0:00:01
   ------------------- -------------------- 430.1/874.7 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 874.7/874.7 kB 3.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [34]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [25]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [64]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [26]:
encoding.decode([1299])

' like'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [2]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)


2 + 2 = 4


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [48]:
pip install groq

   ---------------------------------------- 0.0/143.7 kB ? eta -:--:--
   ------------------------- -------------- 92.2/143.7 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 143.7/143.7 kB 2.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [44]:
pip install openai

In [13]:
from openai import OpenAI

client = OpenAI(
base_url="https://api.groq.com/openai/v1",
api_key="YOUR_GROQ_API_KEY"
)

### A message to OpenAI is a list of dicts

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [26]:
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

"Nice to meet you, Ed! How's your day going so far? Is there something I can help you with or would you like to chat for a bit?"

### OK let's now ask a follow-up question

In [27]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [31]:
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

"I don't know your name. This conversation just started, and you haven't told me anything about yourself yet. Would you like to share your name with me now?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [32]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [33]:
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

'Your name is Ed, as far as our conversation is concerned.'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

